Name Haoxin Ni

Labpartner(s)

In [1]:
#import statements go here


# Class 10.2

Today we are having a guest lecture by Kesse Asante on species distribution modeling in R. 

**W.1** Practice python #13

The Fibonnaci seqence is a sequence of numbers where the next number in the sequence is the sum of the previous two numbers in the sequence. The sequence looks like this: 1, 1, 2, 3, 5, 8, 13, …

Write a program that asks the user how many Fibonnaci numbers to generate and then generates them. Take this opportunity to think about how you can use functions. Make sure to ask the user to enter the number of numbers in the sequence to generate.

In [2]:
def fibonacci(n):
    a, b = 1, 1
    for i in range(n):
        print(a, end=" ")
        a, b = b, a + b

n = int(input("How many Fibonacci numbers? "))
fibonacci(n)

How many Fibonacci numbers?  33


1 1 2 3 5 8 13 21 34 55 89 144 233 377 610 987 1597 2584 4181 6765 10946 17711 28657 46368 75025 121393 196418 317811 514229 832040 1346269 2178309 3524578 

# Lecture 10.2

### Agenda: Asante will have you install some R, then he will walk you through some species distribution modeling

**Please see the pdf document for Lab 10.2 and get the downloads going.**

### Questions

# Lab 10.2

**E.0** Complete the previous labs if you are behind.

**E.1** Asante will give you some exercises to do in R

I used RStudio to fit a BRT model for corals in the Tropical Americas and Caribbean. The study extent is `ext(-100, -55, 0, 32)`. I followed slides 16-25 of the guest lecture, using the six listed environmental layers, a 200 km buffer, a depth range of 0 to 60 m, and 2,000 pseudo-absences.

After removing records with missing environmental values, the model used 451 presences and 2,000 pseudo-absences. I used the spatial block method with 10 folds, as shown on slide 24. The mean test AUC was 0.800.

- [Guest lecture slides](Lab10.2_Guest_Lecture_SDMs.pdf)
- [R script](../Sci_Comp26/Lab10.2_Coral_SDM.R)
- [RStudio screenshot for submission](Lab10.2_outputs/Lab10.2_RStudio_screenshot.png)
- [Spatial cross-validation folds](Lab10.2_outputs/spatial_folds.png)

![Predicted coral distribution in the Tropical Americas and Caribbean](Lab10.2_outputs/coral_prediction.png)

On my map, higher suitability appears along parts of Belize, Cuba, and several Caribbean islands. White ocean areas are outside the modeled mask, so they should not be read as zero suitability.


**E.2** Make notes for yourself on progamming tecniques and commands you learned in the this week, including examples, comments and explainitory text. You can do this here or in a separate notebook that you link to here. Basically, you are making a cheat sheet for yourself.

### R notes

`library()` loads a package for the current session. I only need to install a package once, but I need to load it again when I start a new session.

```r
library(terra)
library(bDSSDMTools)
```

`rast()` reads raster files. `vect()` turns the coordinates into spatial points. `ext()` sets the study area in the order xmin, xmax, ymin, ymax.

```r
studyExtent <- ext(-100, -55, 0, 32)
presences <- vect(corals, geom = c("Lon", "Lat"), crs = "EPSG:4326")
presences <- crop(presences, studyExtent)
```

`studyRegion()` sets the model mask. For the coral exercise, the lecture uses `distanceThreshold = 200000` and `depthPref = c(0, -60)`.

`crop()` cuts the layers down to the study area. `mask()` removes cells outside the chosen region. I also need to remove records where the environmental data are missing.

```r
environmentalLayers <- crop(environmentalLayers, myStudyRegion)
environmentalLayers <- mask(environmentalLayers, myStudyRegion)
```

The coral file only contains presences. I used `pseudoAbsences()` to sample comparison points, then combined the two groups. `PA = 1` means presence and `PA = 0` means pseudo-absence. These zeros are sampled points, not confirmed absences.

```r
set.seed(102)  # Makes the random sampling repeatable.
absences <- pseudoAbsences(environmentalLayers, presences, n = 2000)
```

`generateModelData()` prepares the model data. The lecture uses spatial blocks with 10 folds to split the records by location. `trainModel()` fits the BRT, and `predictModel()` applies it to the environmental layers.

```r
modelData <- generateModelData(records, envConditions = environmentalLayers,
                               method = "blocks", kFolds = 10)
model <- trainModel(modelData, algorithm = "brt")
prediction <- predictModel(model = model, newData = environmentalLayers)
plot(prediction$rasterLayer, col = rev(topo.colors(100)))
```

I saved the plot with `png()` and `dev.off()`. For the submission screenshot, both the script and the predicted map need to be visible in RStudio.
